In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path(__file__).resolve().parents[1])) # Ensure the repo root is on sys.path so sibling packages like `prompt_templates` can be imported


NameError: name '__file__' is not defined

# Testing Supervisor Agent

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
import sys
from pathlib import Path
sys.path.append(str(Path(__file__).resolve().parents[1]))
load_dotenv("config/.env")

from context import DatasetContext
from agents.supervisor import run_supervisor
from structured_outputs import SupervisorOutput, SupervisorAction


# ── Shared context ────────────────────────────────────────────────────────────

context = DatasetContext(
    file_path="data/churn.csv",
    dataset_description=(
        "Monthly snapshot of active mobile subscribers as of billing cycle "
        "close, Q3 2024. Each row is one customer account."
    ),
    column_descriptions={
        "churn":         "1 = churned (bad), 0 = retained",
        "arpu":          "Average revenue per user in USD",
        "tenure_months": "Months the customer has been active",
        "plan_type":     "Subscription tier: prepaid, postpaid, enterprise",
        "credits":       "Billing credits applied — high values indicate disputes",
    },
    business_rules=[
        "credits: high = bad — indicates billing disputes or service failures",
        "churn: 1 = bad outcome, 0 = good",
        "arpu: higher = better",
    ],
    known_issues=[
        "churn column has 3% nulls",
        "enterprise segment has only 45 records — small sample",
    ]
)


# ── Tests ─────────────────────────────────────────────────────────────────────

def test_vague_question():
    """Vague question should return CLARIFY."""
    print("\n" + "="*60)
    print("TEST 1 — Vague question")
    print("="*60)

    result = run_supervisor(
        user_message="can you look at my data",
        context=context,
    )

    print(f"Action:    {result.action}")
    print(f"Reasoning: {result.reasoning}")

    if result.is_clarify():
        print(f"Question:  {result.clarifying_question}")
        print("✓ PASS")
    else:
        print(f"Improved:  {result.improved_question}")
        print("✗ FAIL — expected CLARIFY")


def test_clear_question():
    """Clear question should return HANDOFF with an improved question."""
    print("\n" + "="*60)
    print("TEST 2 — Clear question")
    print("="*60)

    result = run_supervisor(
        user_message="what is the churn rate for each plan type",
        context=context,
    )

    print(f"Action:    {result.action}")
    print(f"Reasoning: {result.reasoning}")

    if result.is_handoff():
        print(f"Improved:\n{result.improved_question}")
        print("✓ PASS")
    else:
        print(f"Question:  {result.clarifying_question}")
        print("✗ FAIL — expected HANDOFF")


def test_business_rules_in_improved_question():
    """Improved question should reference column names and business rules."""
    print("\n" + "="*60)
    print("TEST 3 — Business rules woven into improved question")
    print("="*60)

    result = run_supervisor(
        user_message="compare churned vs retained customers",
        context=context,
    )

    print(f"Action: {result.action}")

    if result.is_handoff():
        print(f"Improved:\n{result.improved_question}")
        has_churn   = "churn"   in result.improved_question.lower()
        has_credits = "credits" in result.improved_question.lower()
        print(f"\nReferences 'churn':   {'✓' if has_churn   else '✗'}")
        print(f"References 'credits': {'✓' if has_credits else '✗'}")
    else:
        print(f"Got CLARIFY: {result.clarifying_question}")


def test_multi_turn():
    """Second message should resolve to HANDOFF after clarification."""
    print("\n" + "="*60)
    print("TEST 4 — Multi-turn conversation")
    print("="*60)

    # First turn
    result1 = run_supervisor(
        user_message="analyse my data",
        context=context,
    )
    print(f"Turn 1 — {result1.action}")
    if result1.is_clarify():
        print(f"Question: {result1.clarifying_question}")

    # Second turn — user answers with a specific question
    history = [
        HumanMessage(content="analyse my data"),
        AIMessage(content=result1.clarifying_question or ""),
    ]

    result2 = run_supervisor(
        user_message="I want to know which plan type has the highest churn rate",
        context=context,
        conversation_history=history,
    )

    print(f"Turn 2 — {result2.action}")
    if result2.is_handoff():
        print(f"Improved: {result2.improved_question}")
        print("✓ PASS")
    else:
        print(f"Still clarifying: {result2.clarifying_question}")


if __name__ == "__main__":
    test_vague_question()
    test_clear_question()
    test_business_rules_in_improved_question()
    test_multi_turn()

# Testing Planner Agent


In [3]:
"""
Note: These tests require a real CSV file at the path in DatasetContext.
Create a small test CSV before running:
"""

import pandas as pd, numpy as np
pd.DataFrame({
    "churn":         np.random.binomial(1, 0.2, 100),
    "arpu":          np.random.normal(55, 15, 100).clip(5),
    "tenure_months": np.random.randint(1, 60, 100),
    "plan_type":     np.random.choice(["prepaid","postpaid","enterprise"], 100),
    "credits":       np.random.exponential(10, 100)
}).to_csv("data/churn.csv", index=False)

In [7]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("config/.env")

from context import DatasetContext
from agents.dataplanner import run_planner
from structured_outputs import PlannerOutput, PlannerAction, AgentType


# ── Create test CSV if it doesn't exist ──────────────────────────────────────

def _ensure_test_csv():
    path = Path("data/churn.csv")
    if not path.exists():
        path.parent.mkdir(exist_ok=True)
        np.random.seed(42)
        n = 200
        pd.DataFrame({
            "churn":         np.random.binomial(1, 0.2, n),
            "arpu":          np.random.normal(55, 15, n).clip(5),
            "tenure_months": np.random.randint(1, 60, n),
            "plan_type":     np.random.choice(
                                 ["prepaid", "postpaid", "enterprise"], n,
                                 p=[0.5, 0.35, 0.15]
                             ),
            "credits":       np.random.exponential(10, n),
        }).to_csv(path, index=False)
        print(f"Created test CSV at {path}")


# ── Shared context ────────────────────────────────────────────────────────────

_ensure_test_csv()

context = DatasetContext(
    file_path="data/churn.csv",
    dataset_description=(
        "Monthly snapshot of active mobile subscribers Q3 2024. "
        "Each row is one customer account."
    ),
    column_descriptions={
        "churn":         "1 = churned (bad), 0 = retained",
        "arpu":          "Average revenue per user in USD",
        "tenure_months": "Months the customer has been active",
        "plan_type":     "Subscription tier: prepaid, postpaid, enterprise",
        "credits":       "Billing credits applied — high values indicate disputes",
    },
    business_rules=[
        "credits: high = bad — indicates billing disputes or service failures",
        "churn: 1 = bad outcome, 0 = good",
        "arpu: higher = better",
    ],
    known_issues=[
        "enterprise segment has only ~30 records — small sample",
    ]
)


# ── Tests ─────────────────────────────────────────────────────────────────────

def test_produces_plan():
    """A clear question should produce a PLAN with steps."""
    print("\n" + "="*60)
    print("TEST 1 — Produces a plan")
    print("="*60)

    result = run_planner(
        improved_question=(
            "What is the churn rate (proportion where churn = 1) broken "
            "down by plan_type? Which segment has the highest churn rate?"
        ),
        context=context,
    )

    print(f"Action: {result.action}")

    if result.is_plan():
        print(f"Goal:   {result.goal}")
        print(f"Steps:  {len(result.steps)}")
        for step in result.steps:
            print(f"  Step {step.id} [{step.agent.value}] — {step.title}")
        print(f"Risks:  {result.risks}")
        print("✓ PASS")
    else:
        print(f"Got CLARIFY: {result.clarifying_question}")
        print("✗ FAIL — expected PLAN")


def test_plan_has_correct_agents():
    """
    A question requiring computation and visualization should assign
    the right agents to the right steps.
    """
    print("\n" + "="*60)
    print("TEST 2 — Correct agents assigned")
    print("="*60)

    result = run_planner(
        improved_question=(
            "What is the churn rate by plan_type? "
            "Show the result as a bar chart."
        ),
        context=context,
    )

    print(f"Action: {result.action}")

    if result.is_plan():
        for step in result.steps:
            print(f"  Step {step.id} [{step.agent.value}] — {step.title}")

        agents_used = {step.agent for step in result.steps}
        has_coder  = AgentType.CODER      in agents_used
        has_viz    = AgentType.VISUALIZER in agents_used

        print(f"\nHas coder step:      {'✓' if has_coder else '✗'}")
        print(f"Has visualizer step: {'✓' if has_viz   else '✗'}")
        print("✓ PASS" if has_coder and has_viz else "✗ FAIL")
    else:
        print(f"Got CLARIFY: {result.clarifying_question}")
        print("✗ FAIL — expected PLAN")


def test_plan_references_real_columns():
    """
    Steps should reference actual column names from the dataset,
    not generic placeholders.
    """
    print("\n" + "="*60)
    print("TEST 3 — Steps reference real column names")
    print("="*60)

    result = run_planner(
        improved_question=(
            "Is there a significant difference in arpu between "
            "churned (churn=1) and retained (churn=0) customers?"
        ),
        context=context,
    )

    print(f"Action: {result.action}")

    if result.is_plan():
        all_inputs = []
        for step in result.steps:
            all_inputs.extend(step.inputs)
            print(f"  Step {step.id} inputs: {step.inputs}")

        has_churn = any("churn" in i.lower() for i in all_inputs)
        has_arpu  = any("arpu"  in i.lower() for i in all_inputs)

        print(f"\nInputs reference 'churn': {'✓' if has_churn else '✗'}")
        print(f"Inputs reference 'arpu':  {'✓' if has_arpu  else '✗'}")
        print("✓ PASS" if has_churn and has_arpu else "✗ FAIL")
    else:
        print(f"Got CLARIFY: {result.clarifying_question}")


def test_business_rules_in_steps():
    """
    Steps involving the credits column should carry the business rule note.
    """
    print("\n" + "="*60)
    print("TEST 4 — Business rules in relevant steps")
    print("="*60)

    result = run_planner(
        improved_question=(
            "Which customers have the highest credits values? "
            "Note that high credits indicate billing disputes (bad signal)."
        ),
        context=context,
    )

    print(f"Action: {result.action}")

    if result.is_plan():
        for step in result.steps:
            print(f"  Step {step.id} — {step.title}")
            print(f"    business_rule_notes: {step.business_rule_notes}")

        has_rule = any(
            step.business_rule_notes and
            ("credit" in step.business_rule_notes.lower() or
             "bad"    in step.business_rule_notes.lower())
            for step in result.steps
        )
        print(f"\nBusiness rule in a step: {'✓' if has_rule else '✗'}")
    else:
        print(f"Got CLARIFY: {result.clarifying_question}")


if __name__ == "__main__":
    test_produces_plan()
    test_plan_has_correct_agents()
    test_plan_references_real_columns()
    test_business_rules_in_steps()

ImportError: cannot import name 'PlannerAction' from 'structured_outputs' (/Users/theodoreleeiv/Documents/GitHub/AI-DATA-ASSISTANT/structured_outputs/__init__.py)